# Bootstrap catalogs, schemas and grants

Creates (if missing) the catalogs and schemas used by the dashboard and the DQ framework, and applies Unity Catalog grants.
Idempotent: safe to run on every deploy. Every action is switched by a job parameter.

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "reporting")
dbutils.widgets.text("source_catalog", "")
dbutils.widgets.text("results_schema", "results")
dbutils.widgets.text("configuration_schema", "configuration")
dbutils.widgets.text("metadata_schema", "metadata")
dbutils.widgets.text("quarantine_schema", "quarantine")
dbutils.widgets.text("owner_group", "")
dbutils.widgets.text("uc_viewer_group", "account users")
dbutils.widgets.text("create_catalog", "false")
dbutils.widgets.text("create_schemas", "true")

catalog          = dbutils.widgets.get("catalog").strip()
schema           = dbutils.widgets.get("schema").strip()
source_catalog   = dbutils.widgets.get("source_catalog").strip() or catalog
results_schema   = dbutils.widgets.get("results_schema").strip()
cfg_schema       = dbutils.widgets.get("configuration_schema").strip()
meta_schema      = dbutils.widgets.get("metadata_schema").strip()
quar_schema      = dbutils.widgets.get("quarantine_schema").strip()
owner_group      = dbutils.widgets.get("owner_group").strip()
uc_viewer_group  = dbutils.widgets.get("uc_viewer_group").strip()
create_catalog   = dbutils.widgets.get("create_catalog").strip().lower() == "true"
create_schemas   = dbutils.widgets.get("create_schemas").strip().lower() == "true"

if not catalog:
    raise ValueError("Parameter 'catalog' is required")
print(f"catalog={catalog}, schema={schema}, source_catalog={source_catalog}")

In [0]:
if create_catalog:
    for cat in sorted({catalog, source_catalog}):
        spark.sql(f"CREATE CATALOG IF NOT EXISTS `{cat}`")
        print(f"Catalog '{cat}' ready")

if create_schemas:
    for cat, sc in [
        (catalog,       schema),
        (source_catalog, results_schema),
        (source_catalog, cfg_schema),
        (source_catalog, meta_schema),
        (source_catalog, quar_schema),
    ]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cat}`.`{sc}`")
        print(f"  `{cat}`.`{sc}` ready")

In [0]:
# Grants are idempotent: re-running them does not change state.
grants = []
if owner_group:
    grants.append((f"GRANT ALL PRIVILEGES ON SCHEMA `{catalog}`.`{schema}` TO `{owner_group}`", f"ALL PRIVILEGES on {catalog}.{schema} -> {owner_group}"))
if uc_viewer_group:
    grants.append((f"GRANT USE CATALOG ON CATALOG `{catalog}` TO `{uc_viewer_group}`", f"USE CATALOG on {catalog} -> {uc_viewer_group}"))
    grants.append((f"GRANT USE SCHEMA ON SCHEMA `{catalog}`.`{schema}` TO `{uc_viewer_group}`", f"USE SCHEMA on {catalog}.{schema} -> {uc_viewer_group}"))
    grants.append((f"GRANT SELECT ON SCHEMA `{catalog}`.`{schema}` TO `{uc_viewer_group}`", f"SELECT on {catalog}.{schema} -> {uc_viewer_group}"))

for sql, label in grants:
    spark.sql(sql)
    print(f"  {label} - ok")

dbutils.notebook.exit("Bootstrap OK")